# DF VAE Roundtrip Debug

This notebook independently runs the full DF-coded VAE roundtrip for one original `edge_depth` sample:

1. Load original `edge_depth` from the dataset
2. Encode it to the DF-coded tensor
3. Run the selected VAE encode and decode
4. Decode the DF tensor back to `edge_depth`
5. Compute abs error, MSE, and L1 on GT valid pixels
6. Visualize each hit and export the point clouds as `.ply`

The notebook reuses the existing `edge_vae` modules instead of re-implementing the transforms or the VAE wrapper.

DF valid depths now use the same geometry normalization factor as raw mode: `1 / sqrt(3)`.

Set `vae_choice` in cell 3 to one of `sdxl`, `flux`, or `sd3` before running the notebook.


In [ ]:
from pathlib import Path
import importlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display

repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import edge_vae.configs as edge_vae_configs
import edge_vae.transforms as edge_vae_transforms

from edge3d.tensor_format import load_sample_modalities
from edge_vae.codec import DiffusersVAECodec
from edge_vae.visualization import save_roundtrip_pointclouds

apply_edge_vae_overrides = edge_vae_configs.apply_edge_vae_overrides
load_edge_vae_config = edge_vae_configs.load_edge_vae_config
EDGE_DEPTH_NORMALIZATION_SCALE = edge_vae_transforms.EDGE_DEPTH_NORMALIZATION_SCALE
decode_df_tensor_to_edge_depth = edge_vae_transforms.decode_df_tensor_to_edge_depth
decode_raw_tensor_to_edge_depth = edge_vae_transforms.decode_raw_tensor_to_edge_depth
encode_edge_depth_to_df_tensor = edge_vae_transforms.encode_edge_depth_to_df_tensor
encode_edge_depth_to_raw_tensor = edge_vae_transforms.encode_edge_depth_to_raw_tensor

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['image.cmap'] = 'viridis'


In [ ]:
sample_id = '07ab702adb6749858bce3b499316d7be'
vae_presets = {
    'sdxl': {
        'pretrained_model_name_or_path': 'madebyollin/sdxl-vae-fp16-fix',
        'torch_dtype': 'float16',
    },
    'flux': {
        'pretrained_model_name_or_path': 'black-forest-labs/FLUX.1-schnell',
        'subfolder': 'vae',
        'torch_dtype': 'float16',
    },
    'sd3': {
        'pretrained_model_name_or_path': 'stabilityai/stable-diffusion-3-medium-diffusers',
        'subfolder': 'vae',
        'torch_dtype': 'float16',
    },
}
vae_choice = 'flux'  # switch to 'flux' or 'sd3'
if vae_choice not in vae_presets:
    raise ValueError(f'Unsupported vae_choice {vae_choice!r}. Expected one of {sorted(vae_presets)}')

sample_path = Path('/home/devdata/edge3d_data/equirectangular_data') / f'{sample_id}.npz'
output_dir = repo_root / 'runs' / 'edge_vae_notebook_debug' / 'df' / vae_choice / sample_id
output_dir.mkdir(parents=True, exist_ok=True)

config = load_edge_vae_config(repo_root / 'configs' / 'vae' / 'edge_df_sdxl_roundtrip.yaml', mode='df')
config = apply_edge_vae_overrides(config, device='cuda')
vae_cfg = dict(config['vae'])
vae_cfg.update(vae_presets[vae_choice])
transform_cfg = dict(config['transform'])
runtime_cfg = dict(config['runtime'])

print('sample_path =', sample_path)
print('output_dir =', output_dir)
print('vae_choice =', vae_choice)
print('edge_depth_normalization_scale =', EDGE_DEPTH_NORMALIZATION_SCALE)
print(json.dumps({'vae': vae_cfg, 'transform': transform_cfg, 'runtime': runtime_cfg}, indent=2))


In [ ]:
raw_probe = np.array(
    [
        [[0.0, 0.5], [1.0, 0.0]],
        [[0.25, 0.0], [0.0, 0.75]],
        [[0.0, 0.0], [0.5, 0.25]],
    ],
    dtype=np.float32,
)
raw_encoded_probe = encode_edge_depth_to_raw_tensor(raw_probe, raw_scale=EDGE_DEPTH_NORMALIZATION_SCALE)
raw_decoded_probe = decode_raw_tensor_to_edge_depth(
    raw_encoded_probe,
    raw_scale=EDGE_DEPTH_NORMALIZATION_SCALE,
    valid_threshold=0.0,
)

print({
    'edge_depth_normalization_scale': EDGE_DEPTH_NORMALIZATION_SCALE,
    'raw_encoded_min': float(raw_encoded_probe.min()),
    'raw_encoded_max': float(raw_encoded_probe.max()),
    'raw_roundtrip_allclose': bool(np.allclose(raw_decoded_probe, raw_probe, atol=1.0e-6)),
})


In [ ]:
payload = load_sample_modalities(sample_path, decode_model_normal=False)
gt_edge_depth = np.asarray(payload['edge_depth'], dtype=np.float32)
valid_eps = float(transform_cfg['valid_eps'])
valid_mask = np.isfinite(gt_edge_depth) & (gt_edge_depth > valid_eps)

input_df_tensor = encode_edge_depth_to_df_tensor(
    gt_edge_depth,
    beta=float(transform_cfg['beta']),
    valid_eps=valid_eps,
)

summary = []
for hit_idx in range(gt_edge_depth.shape[0]):
    hit_valid = valid_mask[hit_idx]
    gt_values = gt_edge_depth[hit_idx][hit_valid]
    df_values = input_df_tensor[hit_idx]
    summary.append({
        'hit': hit_idx,
        'gt_valid_count': int(hit_valid.sum()),
        'gt_valid_min': float(gt_values.min()) if gt_values.size else None,
        'gt_valid_max': float(gt_values.max()) if gt_values.size else None,
        'df_min': float(df_values.min()),
        'df_max': float(df_values.max()),
    })

print('edge_depth_normalization_scale =', EDGE_DEPTH_NORMALIZATION_SCALE)
print(json.dumps(summary, indent=2))
torch.save(torch.from_numpy(input_df_tensor), output_dir / 'input_df_encoded.pt')
print('saved', output_dir / 'input_df_encoded.pt')


In [ ]:
codec = DiffusersVAECodec(
    pretrained_model_name_or_path=str(vae_cfg['pretrained_model_name_or_path']),
    torch_dtype=str(vae_cfg['torch_dtype']),
    device=str(runtime_cfg['device']),
    subfolder=str(vae_cfg['subfolder']) if 'subfolder' in vae_cfg else None,
)

input_tensor = torch.from_numpy(input_df_tensor[None, ...])
latents = codec.encode(input_tensor)
decoded_df_tensor = codec.decode(latents)[0].detach().to(dtype=torch.float32).cpu().numpy()
decoded_edge_depth = decode_df_tensor_to_edge_depth(
    decoded_df_tensor,
    valid_threshold=float(transform_cfg['decode_valid_threshold']),
)

torch.save(latents.detach().to(dtype=torch.float32).cpu(), output_dir / 'latents.pt')
torch.save(torch.from_numpy(decoded_df_tensor), output_dir / 'decoded_df_encoded.pt')
torch.save(torch.from_numpy(decoded_edge_depth), output_dir / 'decoded_edge_depth.pt')

print('vae_choice =', vae_choice)
print('selected_vae =', json.dumps(vae_cfg, indent=2))
print('edge_depth_normalization_scale =', EDGE_DEPTH_NORMALIZATION_SCALE)
print('resolved_device =', codec.device)
print('resolved_torch_dtype =', codec.torch_dtype)
print('latents.shape =', tuple(latents.shape))
print('decoded_df_tensor range =', float(decoded_df_tensor.min()), float(decoded_df_tensor.max()))
print('decoded_edge_depth range =', float(decoded_edge_depth.min()), float(decoded_edge_depth.max()))


In [ ]:
fig, ax = plt.subplots(1, 3)
ax[0].imshow(input_df_tensor[0], vmin=-1, vmax=1.0)
ax[0].set_title('input_df_tensor (encoded)')
ax[1].imshow(decoded_df_tensor[0], vmin=-1, vmax=1.0)
ax[1].set_title('decoded_df_tensor (decoded)')
diff = np.abs(decoded_df_tensor - input_df_tensor)
ax[2].imshow(diff[0], vmin=0, vmax=1)
ax[2].set_title('absolute difference')
plt.show()

In [ ]:
abs_error = np.abs(decoded_edge_depth - gt_edge_depth).astype(np.float32)
diff_valid = decoded_edge_depth[valid_mask] - gt_edge_depth[valid_mask]
mse_valid = float(np.mean(np.square(diff_valid)))
l1_valid = float(np.mean(np.abs(diff_valid)))
rmse_valid = float(np.sqrt(mse_valid))

per_hit_metrics = []
for hit_idx in range(gt_edge_depth.shape[0]):
    hit_valid = valid_mask[hit_idx]
    hit_diff = decoded_edge_depth[hit_idx][hit_valid] - gt_edge_depth[hit_idx][hit_valid]
    per_hit_metrics.append({
        'hit': hit_idx,
        'valid_count': int(hit_valid.sum()),
        'mse': float(np.mean(np.square(hit_diff))),
        'l1': float(np.mean(np.abs(hit_diff))),
        'rmse': float(np.sqrt(np.mean(np.square(hit_diff)))),
    })

metrics = {
    'valid_pixel_count': int(valid_mask.sum()),
    'mse_valid': mse_valid,
    'l1_valid': l1_valid,
    'rmse_valid': rmse_valid,
    'per_hit': per_hit_metrics,
}

with (output_dir / 'metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(metrics, handle, indent=2)

print(json.dumps(metrics, indent=2))

In [ ]:
def plot_hit_panels(hit_idx: int) -> None:
    hit_gt = gt_edge_depth[hit_idx]
    hit_input_df = input_df_tensor[hit_idx]
    hit_decoded_df = decoded_df_tensor[hit_idx]
    hit_decoded_depth = decoded_edge_depth[hit_idx]
    hit_abs_error = abs_error[hit_idx]
    hit_threshold_mask = (hit_decoded_df > float(transform_cfg['decode_valid_threshold'])).astype(np.float32)

    depth_vmax = max(float(np.nanmax(hit_gt)), float(np.nanmax(hit_decoded_depth)), 1e-6)
    error_vmax = max(float(np.nanmax(hit_abs_error)), 1e-6)

    fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=120)
    panels = [
        (hit_gt, f'Original edge depth hit{hit_idx}', 'viridis', 0.0, depth_vmax),
        (hit_input_df, f'Input DF tensor hit{hit_idx}', 'coolwarm', -1.0, 1.0),
        (hit_decoded_df, f'Decoded DF tensor hit{hit_idx}', 'coolwarm', -1.0, 1.0),
        (hit_decoded_depth, f'Decoded edge depth hit{hit_idx}', 'viridis', 0.0, depth_vmax),
        (hit_abs_error, f'Abs error hit{hit_idx}', 'magma', 0.0, error_vmax),
        (hit_threshold_mask, f'Threshold mask hit{hit_idx}', 'gray', 0.0, 1.0),
    ]

    for ax, (image, title, cmap, vmin, vmax) in zip(axes.flat, panels, strict=True):
        ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])

    fig.tight_layout()
    fig.savefig(output_dir / f'hit{hit_idx}_panels.png')
    plt.show()
    plt.close(fig)

for hit_idx in range(gt_edge_depth.shape[0]):
    plot_hit_panels(hit_idx)

In [ ]:
pointcloud_paths = save_roundtrip_pointclouds(
    output_dir,
    target_edge_depth=gt_edge_depth,
    decoded_edge_depth=decoded_edge_depth,
)
print(json.dumps(pointcloud_paths, indent=2))

In [ ]:
saved_files = sorted(path.name for path in output_dir.iterdir())
print('Saved files:')
for name in saved_files:
    print(' -', name)